# Energy Forecast — Prediction Performance (last 30 days)

Pulls live data from Home Assistant via SMB, fetches weather from Open-Meteo, and renders accuracy charts for the last 30 days.

**Pre-requisite:** `SMB_PASSWORD` environment variable must be set before starting the kernel.


In [ ]:
from __future__ import annotations

import io
import json
import os
from datetime import date, timedelta

import altair as alt
import pandas as pd
import requests
import yaml
from smb.SMBConnection import SMBConnection


In [ ]:
# ── Credentials ──────────────────────────────────────────────────────────────
SMB_USER = os.getenv("SMB_USER", "martin")
SMB_PASSWORD = os.getenv("SMB_PASSWORD")
if not SMB_PASSWORD:
    raise RuntimeError(
        "SMB_PASSWORD environment variable is not set. "
        "Set it before starting the kernel: export SMB_PASSWORD=<password>"
    )

# ── Constants ─────────────────────────────────────────────────────────────────
HA_HOST = "homeassistant"
SMB_SHARE = "addon_configs"
AD_BASE = "a0d7b954_appdaemon/apps"
FORECAST_REMOTE = f"{AD_BASE}/energy_forecast"

TZ = "Europe/Zurich"
CUTOFF_DAYS = 30
EV_THRESHOLD_KWH = 7.0  # matches EV_CHARGING_THRESHOLD_KWH in const.py

WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print("Config OK — SMB_PASSWORD is set")

## 1 · Fetch data from Home Assistant (SMB)

In [ ]:
def _smb_read(remote_path: str) -> bytes:
    conn = SMBConnection(SMB_USER, SMB_PASSWORD, "notebook", HA_HOST, use_ntlm_v2=True)
    if not conn.connect(HA_HOST, 445):
        raise ConnectionError(f"SMB connection to {HA_HOST}:445 failed")
    try:
        buf = io.BytesIO()
        conn.retrieveFile(SMB_SHARE, remote_path, buf)
        return buf.getvalue()
    finally:
        conn.close()


# Pull files — wrap in try/except so downstream cells get empty dicts on failure
try:
    print("Fetching pred_history.json …")
    _pred_raw = json.loads(_smb_read(f"{FORECAST_REMOTE}/pred_history.json"))
    print(f"  pred entries : {len(_pred_raw.get('pred', {}))}")
    print(f"  actual entries: {len(_pred_raw.get('actuals', {}))}")

    print("Fetching apps.yaml …")
    _apps_yaml = yaml.safe_load(_smb_read(f"{AD_BASE}/apps.yaml"))
except Exception as _smb_exc:
    print(f"WARNING: SMB fetch failed ({_smb_exc}). Charts will be empty.")
    _pred_raw = {}
    _apps_yaml = {}

if not _pred_raw.get("pred"):
    print("WARNING: pred_history.json has 0 prediction entries — charts will be empty.")
    print("The app needs at least one update cycle on the HA system to populate this file.")


## 2 · Fetch outdoor temperature from Open-Meteo archive

In [ ]:
# Read lat/lon from apps.yaml; fall back to Zurich city centre
_ef_cfg = _apps_yaml.get("energy_forecast", {})
_lat = _ef_cfg.get("latitude", 47.376)
_lon = _ef_cfg.get("longitude", 8.541)

_start = (date.today() - timedelta(days=CUTOFF_DAYS)).isoformat()
_end = date.today().isoformat()

_url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={_lat}&longitude={_lon}"
    f"&start_date={_start}&end_date={_end}"
    f"&hourly=temperature_2m&timezone=Europe%2FZurich"
)

try:
    _resp = requests.get(_url, timeout=30)
    _resp.raise_for_status()
    _weather = _resp.json()
    temp_df = pd.DataFrame({
        "timestamp": pd.to_datetime(_weather["hourly"]["time"]),
        "temp_c": _weather["hourly"]["temperature_2m"],
    })
    temp_df["timestamp"] = temp_df["timestamp"].dt.floor("h")
    print(f"Weather fetched: {len(temp_df)} hourly rows, {_start} → {_end}")
except Exception as exc:
    temp_df = pd.DataFrame(columns=["timestamp", "temp_c"])
    print(f"WARNING: Open-Meteo fetch failed ({exc}). Chart 6 (temp correlation) will be skipped.")


## 3 · Data preparation

In [ ]:
def _history_to_df(d: dict, col: str) -> pd.DataFrame:
    rows = [(pd.Timestamp(ts), float(v)) for ts, v in d.items()]
    if not rows:
        return pd.DataFrame(columns=["timestamp", col])
    df = pd.DataFrame(rows, columns=["timestamp", col])
    df["timestamp"] = df["timestamp"].dt.floor("h")
    return df.sort_values("timestamp").reset_index(drop=True)


pred_df = _history_to_df(_pred_raw.get("pred", {}), "pred_kwh")
actuals_df = _history_to_df(_pred_raw.get("actuals", {}), "actual_kwh")

# Merge on matched hours only
error_df = pd.merge(pred_df, actuals_df, on="timestamp", how="inner")

# Filter to last 30 days
_cutoff = pd.Timestamp.now(tz=TZ).tz_localize(None) - pd.Timedelta(days=CUTOFF_DAYS)
error_df = error_df[error_df["timestamp"] >= _cutoff].copy()

# Derived columns
error_df["error"] = error_df["pred_kwh"] - error_df["actual_kwh"]
error_df["abs_error"] = error_df["error"].abs()
error_df["is_ev"] = error_df["actual_kwh"] > EV_THRESHOLD_KWH
error_df["date"] = pd.to_datetime(error_df["timestamp"].dt.date)
error_df["hour"] = error_df["timestamp"].dt.hour
error_df["weekday"] = error_df["timestamp"].dt.day_name()

# Merge outdoor temperature
error_df = error_df.merge(temp_df, on="timestamp", how="left")

# Daily aggregates
daily_df = (
    error_df.groupby("date", as_index=False)
    .agg(daily_pred=("pred_kwh", "sum"), daily_actual=("actual_kwh", "sum"), daily_mae=("abs_error", "mean"))
    .sort_values("date")
    .reset_index(drop=True)
)
daily_df["rolling_mae_7d"] = daily_df["daily_mae"].rolling(7, min_periods=1).mean()

print(f"error_df : {len(error_df)} hourly rows, {error_df['timestamp'].min()} → {error_df['timestamp'].max()}")
print(f"daily_df : {len(daily_df)} days")
print(f"EV hours : {error_df['is_ev'].sum()}")
print(f"Overall MAE: {error_df['abs_error'].mean():.3f} kWh")
print(f"Temp coverage: {error_df['temp_c'].notna().sum()}/{len(error_df)} hours")

if error_df.empty:
    print("\nWARNING: No matched pred/actual pairs in the last 30 days. Charts will be empty.")


## 4 · Charts

### Chart 1 — Daily MAE trend

Mean absolute error per day. The dashed red line is a 7-day rolling average.


In [ ]:
_line_daily = (
    alt.Chart(daily_df)
    .mark_line(point=True, color="steelblue")
    .encode(
        x=alt.X("date:T", title="Date"),
        y=alt.Y("daily_mae:Q", title="MAE (kWh)", scale=alt.Scale(zero=True)),
        tooltip=[alt.Tooltip("date:T", title="Date"), alt.Tooltip("daily_mae:Q", title="MAE (kWh)", format=".3f")],
    )
)

_line_rolling = (
    alt.Chart(daily_df)
    .mark_line(strokeDash=[6, 3], color="crimson")
    .encode(
        x=alt.X("date:T"),
        y=alt.Y("rolling_mae_7d:Q"),
        tooltip=[alt.Tooltip("date:T", title="Date"), alt.Tooltip("rolling_mae_7d:Q", title="7d rolling MAE", format=".3f")],
    )
)

chart1 = (_line_daily + _line_rolling).properties(
    title="Daily MAE — last 30 days  (blue = daily, red dashed = 7-day rolling avg)",
    width=700,
    height=300,
)
chart1


### Chart 2 — Daily totals: Predicted vs Actual

Side-by-side bars of summed predicted and actual kWh per day.

In [ ]:
_daily_long = daily_df.melt(
    id_vars=["date"],
    value_vars=["daily_pred", "daily_actual"],
    var_name="series",
    value_name="kwh",
)
_daily_long["series"] = _daily_long["series"].map({"daily_pred": "Predicted", "daily_actual": "Actual"})

chart2 = (
    alt.Chart(_daily_long)
    .mark_bar()
    .encode(
        x=alt.X("date:T", title="Date"),
        y=alt.Y("kwh:Q", title="kWh"),
        color=alt.Color(
            "series:N",
            scale=alt.Scale(domain=["Predicted", "Actual"], range=["steelblue", "orange"]),
            legend=alt.Legend(title="Series"),
        ),
        xOffset=alt.XOffset("series:N"),
        tooltip=[alt.Tooltip("date:T", title="Date"), "series:N", alt.Tooltip("kwh:Q", format=".2f")],
    )
    .properties(title="Daily totals — Predicted vs Actual", width=700, height=300)
)
chart2


### Chart 3 — Predicted vs Actual (per hour)

Each point is one hour. The dashed diagonal is perfect prediction. Red points are EV charging hours (actual > 7 kWh).


In [ ]:
if error_df.empty:
    _max_kwh = 1.0
else:
    _max_kwh = float(max(error_df["actual_kwh"].max(), error_df["pred_kwh"].max())) * 1.05

_scatter = (
    alt.Chart(error_df)
    .mark_point(opacity=0.45, size=30)
    .encode(
        x=alt.X("actual_kwh:Q", title="Actual (kWh)", scale=alt.Scale(domain=[0, _max_kwh])),
        y=alt.Y("pred_kwh:Q", title="Predicted (kWh)", scale=alt.Scale(domain=[0, _max_kwh])),
        color=alt.Color(
            "is_ev:N",
            scale=alt.Scale(domain=[False, True], range=["steelblue", "crimson"]),
            legend=alt.Legend(title="EV hour"),
        ),
        tooltip=[
            alt.Tooltip("timestamp:T", title="Hour"),
            alt.Tooltip("actual_kwh:Q", format=".3f", title="Actual (kWh)"),
            alt.Tooltip("pred_kwh:Q", format=".3f", title="Predicted (kWh)"),
            "is_ev:N",
        ],
    )
)

_ref_line = (
    alt.Chart(pd.DataFrame({"v": [0, _max_kwh]}))
    .mark_line(color="black", strokeDash=[5, 3])
    .encode(x=alt.X("v:Q"), y=alt.Y("v:Q"))
)

chart3 = (_scatter + _ref_line).properties(
    title="Predicted vs Actual (per hour) — red = EV charging hours",
    width=700,
    height=400,
)
chart3


### Chart 4 — Error heatmap by hour × weekday

Mean absolute error for each hour-of-day / weekday combination. Darker red = higher average error.


In [ ]:
_heatmap_df = (
    error_df.groupby(["hour", "weekday"], as_index=False)["abs_error"].mean()
)

chart4 = (
    alt.Chart(_heatmap_df)
    .mark_rect()
    .encode(
        x=alt.X("hour:O", title="Hour of day (0–23)"),
        y=alt.Y("weekday:O", sort=WEEKDAY_ORDER, title="Day of week"),
        color=alt.Color(
            "abs_error:Q",
            scale=alt.Scale(scheme="reds"),
            title="Mean abs error (kWh)",
        ),
        tooltip=[
            "hour:O",
            "weekday:N",
            alt.Tooltip("abs_error:Q", format=".3f", title="Mean abs error (kWh)"),
        ],
    )
    .properties(title="Mean absolute error by hour × weekday", width=700, height=220)
)
chart4


### Chart 5 — Error distribution

Distribution of `predicted − actual` across all hourly pairs. A distribution centred at 0 means no systematic bias. A right shift means the model over-predicts; left shift means under-prediction.


In [ ]:
_hist = (
    alt.Chart(error_df)
    .mark_bar()
    .encode(
        x=alt.X("error:Q", bin=alt.Bin(step=0.1), title="Error (predicted − actual, kWh)"),
        y=alt.Y("count()", title="Count"),
        tooltip=["count()"],
    )
)

_zero_rule = (
    alt.Chart(pd.DataFrame({"x": [0.0]}))
    .mark_rule(color="crimson", strokeDash=[5, 3], strokeWidth=2)
    .encode(x=alt.X("x:Q"))
)

chart5 = (_hist + _zero_rule).properties(
    title="Error distribution (predicted − actual)  — red line = zero",
    width=700,
    height=300,
)
chart5


### Chart 6 — Absolute error vs outdoor temperature

Shows whether forecast errors are correlated with temperature. A rising trend on the left (cold) or right (hot) side suggests seasonal bias. Red = EV charging hours.


In [ ]:
_temp_error_df = error_df.dropna(subset=["temp_c"])

if _temp_error_df.empty:
    print("Chart 6 skipped — no temperature data available (Open-Meteo fetch failed).")
else:
    chart6 = (
        alt.Chart(_temp_error_df)
        .mark_point(opacity=0.4, size=28)
        .encode(
            x=alt.X("temp_c:Q", title="Outdoor temperature (°C)"),
            y=alt.Y("abs_error:Q", title="Absolute error (kWh)", scale=alt.Scale(zero=True)),
            color=alt.Color(
                "is_ev:N",
                scale=alt.Scale(domain=[False, True], range=["steelblue", "crimson"]),
                legend=alt.Legend(title="EV hour"),
            ),
            tooltip=[
                alt.Tooltip("timestamp:T", title="Hour"),
                alt.Tooltip("temp_c:Q", format=".1f", title="Temp (°C)"),
                alt.Tooltip("abs_error:Q", format=".3f", title="Abs error (kWh)"),
                "is_ev:N",
            ],
        )
        .properties(
            title="Absolute error vs outdoor temperature — red = EV charging hours",
            width=700,
            height=300,
        )
    )
    chart6
